# 🛒 Voice Cart — Intent Classifier Training

This notebook trains a lightweight intent classifier on top of MiniLM-L12-v2 sentence embeddings.

**Pipeline:**
1. Load `dataset.json` (1600 multilingual phrases)
2. Encode each phrase with `paraphrase-multilingual-MiniLM-L12-v2` → 384-dim vectors
3. Train a Dense classifier: 384 → 128 → 64 → 4 (softmax)
4. Export to TensorFlow.js format

**Runtime:** Select GPU (Runtime → Change runtime type → T4 GPU)

In [ ]:
!pip install -q sentence-transformers tensorflowjs tensorflow numpy scikit-learn

In [ ]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflowjs as tfjs

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

## Step 1 — Upload & Load Dataset

Upload your `dataset.json` file using the file upload button below, or mount Google Drive.

In [ ]:
from google.colab import files

# Option 1: Upload directly
uploaded = files.upload()  # Upload dataset.json

# Load the dataset
with open('dataset.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print(f'Loaded {len(dataset)} phrases')

# Show distribution
from collections import Counter
label_counts = Counter(item['label'] for item in dataset)
print(f'\nLabel distribution:')
for label, count in sorted(label_counts.items()):
    print(f'  {label}: {count}')

In [ ]:
# Extract texts and labels
texts = [item['text'] for item in dataset]
labels = [item['label'] for item in dataset]

# Encode labels
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

# IMPORTANT: Print the label order — this must match the app's INTENT_LABELS
print(f'Label order: {list(label_encoder.classes_)}')
print(f'This MUST match INTENT_LABELS in the app!')
print(f'\nTotal samples: {len(texts)}')

## Step 2 — Encode with MiniLM-L12-v2

This encodes all phrases into 384-dimensional semantic vectors using the same model that runs in the browser via Transformers.js.

In [ ]:
# Load the multilingual sentence encoder
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f'Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}')

# Encode all phrases
print(f'Encoding {len(texts)} phrases...')
embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)
embeddings = np.array(embeddings)
print(f'Embeddings shape: {embeddings.shape}')  # Should be (N, 384)

In [ ]:
# Verify: similar phrases should have similar embeddings
test_phrases = [
    "Add milk to my list",
    "doodh daal do",
    "paal add pannu",
    "Remove bananas",
    "kela hatao"
]
test_embeddings = model.encode(test_phrases)

from sklearn.metrics.pairwise import cosine_similarity
sim_matrix = cosine_similarity(test_embeddings)

print('Cosine similarity matrix (similar phrases should have high similarity):')
for i, p in enumerate(test_phrases):
    print(f'  {p[:30]:30s}', [f'{s:.2f}' for s in sim_matrix[i]])

## Step 3 — Train the Classifier

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, encoded_labels, test_size=0.15, random_state=42, stratify=encoded_labels
)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

# One-hot encode labels for training
num_classes = len(label_encoder.classes_)
y_train_onehot = keras.utils.to_categorical(y_train, num_classes)
y_test_onehot = keras.utils.to_categorical(y_test, num_classes)

# Build the classifier
classifier = keras.Sequential([
    layers.Input(shape=(384,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation='softmax')
])

classifier.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

classifier.summary()

In [ ]:
# Train
history = classifier.fit(
    X_train, y_train_onehot,
    validation_data=(X_test, y_test_onehot),
    epochs=50,
    batch_size=32,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5)
    ]
)

# Evaluate
loss, accuracy = classifier.evaluate(X_test, y_test_onehot)
print(f'\nTest accuracy: {accuracy:.4f}')
print(f'Test loss: {loss:.4f}')

In [ ]:
# Detailed evaluation
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_pred = classifier.predict(X_test).argmax(axis=1)

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Intent Classifier — Confusion Matrix')
plt.tight_layout()
plt.show()

## Step 4 — Test on Unseen Multilingual Phrases

In [ ]:
# Test on completely new phrases
test_inputs = [
    # Add intent
    ("add 2 kg onions", "add"),
    ("pyaaz daal do", "add"),
    ("doodh lana hai 2 litre", "add"),
    ("vengayam add pannu", "add"),
    ("kanda beku", "add"),
    # Remove intent
    ("remove the bread", "remove"),
    ("paneer hatao list se", "remove"),
    ("delete tomatoes", "remove"),
    # Search intent
    ("find organic apples", "search"),
    ("toothpaste dikhao under 100", "search"),
    ("show me all dairy items", "search"),
    # Update qty intent
    ("change milk to 3 litres", "update_qty"),
    ("update rice quantity to 5 kg", "update_qty"),
    ("doodh 2 litre kar do", "update_qty"),
]

test_embs = model.encode([t[0] for t in test_inputs])
preds = classifier.predict(test_embs)

print(f'{"Phrase":45s} {"Expected":12s} {"Predicted":12s} {"Conf":6s} {"✓":3s}')
print('-' * 80)
for (phrase, expected), pred_probs in zip(test_inputs, preds):
    pred_idx = pred_probs.argmax()
    pred_label = label_encoder.classes_[pred_idx]
    confidence = pred_probs[pred_idx]
    match = '✓' if pred_label == expected else '✗'
    print(f'{phrase:45s} {expected:12s} {pred_label:12s} {confidence:.3f}  {match}')

## Step 5 — Export to TensorFlow.js

In [ ]:
import os

# Export to TF.js format
export_dir = 'tfjs_model'
os.makedirs(export_dir, exist_ok=True)

tfjs.converters.save_keras_model(classifier, export_dir)

print(f'\nExported model files:')
for f in os.listdir(export_dir):
    size = os.path.getsize(os.path.join(export_dir, f))
    print(f'  {f} ({size:,} bytes)')

# IMPORTANT: Save the label order
label_order = list(label_encoder.classes_)
print(f'\n⚠️  INTENT_LABELS must be set to: {label_order}')
print('Update src/constants/index.ts if the order differs!')

In [ ]:
# Download the exported model files
import shutil

# Create a zip for easy download
shutil.make_archive('voice_cart_model', 'zip', export_dir)
files.download('voice_cart_model.zip')

print('\n✅ Download complete!')
print('\nNext steps:')
print('1. Unzip voice_cart_model.zip')
print('2. Copy model.json and *.bin files to public/model/ in your project')
print('3. The app will load the model from /model/model.json at runtime')

## Training History Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()